In [ ]:
# ============================================================
# STEERING INFERENCE — ALPHA GRID (FULL, FINAL + DELTA BLEU)
# ============================================================

import re
import numpy as np
from pathlib import Path

import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# ------------------------------------------------------------
# GLOBAL SETTINGS
# ------------------------------------------------------------
torch.set_grad_enabled(False)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TORCH_DTYPE = torch.float16

# ALPHAS = [0, 1]
ALPHAS = [-1.0, -0.5, 0.0, 0.5, 1.0, 1.5]
MAX_NEW_TOKENS = 2000


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

#ROOT_DIR is a root directory of the project. Put the path instead of "...".
ROOT_DIR = Path(r"...")

MODEL_DIR = ROOT_DIR / "models" / "MODEL" # choose the model from the models directory
TEST_PATH = ROOT_DIR / "data" / "datasets" / "separable" / "seperable_test.xlsx"
STEERING_PATH = ROOT_DIR / "steering" / "separable" / "steering_separable_Reasoner_Zero.pt" #choose separable steering vector file for the chosen model

OUT_PATH = ROOT_DIR / "RESULT.xlsx" #you can change the name and path of the output file or keep the default one

# ------------------------------------------------------------
# NORMALIZATION + BLEU
# ------------------------------------------------------------
def normalize_answer(s: str) -> str:
    if not s:
        return ""
    s = re.sub(r"\\boxed\{(.+?)\}", r"\1", s)
    s = re.sub(r"^y\s*=\s*", "", s)
    s = s.replace("\\left", "").replace("\\right", "")
    s = s.replace(" ", "")
    return s


def tokenize_math(expr: str):
    return re.findall(r"[A-Za-z]+|\d+|\^|\+|\-|\*|\/|\(|\)|\{|\}|C", expr)


def compute_bleu(true: str, pred: str) -> float:
    if not true or not pred:
        return 0.0
    return sentence_bleu(
        [tokenize_math(true)],
        tokenize_math(pred),
        weights=(0.5, 0.5),
        smoothing_function=SmoothingFunction().method1
    )


# ------------------------------------------------------------
# BOX EXTRACTION
# ------------------------------------------------------------
def extract_boxed(text: str) -> str:
    matches = [m.start() for m in re.finditer(r"\\boxed\{", text)]
    if not matches:
        return ""
    start = matches[-1] + len(r"\boxed{")
    depth, i = 1, start
    while i < len(text) and depth:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    return text[start:i-1].strip()


# ------------------------------------------------------------
# PROMPT
# ------------------------------------------------------------
BASE_SYS = """
You are a symbolic mathematics model.

Task: compute y(x) from the given derivative y'(x).

Output:
- ONLY final answer
- LaTeX
- \\boxed{y=...+C}
"""


def make_prompt(eq: str) -> str:
    return BASE_SYS + f"\nPROBLEM:\n{eq}\n\nANSWER:\n"


# ------------------------------------------------------------
# MODEL + TOKENIZER
# ------------------------------------------------------------
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=TORCH_DTYPE,
    device_map={"": 0} if DEVICE.type == "cuda" else None,
)

model.eval()

print("Model loaded.")


# ------------------------------------------------------------
# STEERING MODULE (HOOKS ON ALL LAYERS)
# ------------------------------------------------------------
class Steering:
    def __init__(self, model):
        self.layers = model.model.layers
        self.handles = []
        self.alpha = 0.0

        data = torch.load(STEERING_PATH, map_location=DEVICE)
        self.vectors = data["vectors"].to(DEVICE).to(TORCH_DTYPE)

    def install(self):
        self.remove()

        def make_hook(i):
            v = self.vectors[i]

            def hook(_, __, out):
                if isinstance(out, tuple):
                    out = out[0]
                return out + self.alpha * v

            return hook

        for i, layer in enumerate(self.layers):
            handle = layer.mlp.down_proj.register_forward_hook(make_hook(i))
            self.handles.append(handle)

    def remove(self):
        for h in self.handles:
            try:
                h.remove()
            except:
                pass
        self.handles = []


steering = Steering(model)
steering.install()


# ------------------------------------------------------------
# DATA
# ------------------------------------------------------------
print("Loading dataset...")

df = pd.read_excel(TEST_PATH)
df = df.dropna(subset=["equation", "true_answer"]).reset_index(drop=True)

print(f"Dataset size: {len(df)}")


# ------------------------------------------------------------
# AUTOCast CONTEXT
# ------------------------------------------------------------
if DEVICE.type == "cuda":
    autocast_ctx = torch.amp.autocast(device_type="cuda")
else:
    autocast_ctx = torch.no_grad()


# ------------------------------------------------------------
# INFERENCE LOOP
# ------------------------------------------------------------
print("Starting inference...")

results = []

for row in tqdm(df.itertuples(index=False), total=len(df)):

    eq = str(row.equation)
    true = str(row.true_answer)

    prompt = make_prompt(eq)
    enc = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    bleu_0 = None  # baseline BLEU (alpha=0)

    for alpha in ALPHAS:
        steering.alpha = float(alpha)

        with torch.inference_mode(), autocast_ctx:
            seq = model.generate(
                **enc,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        text = tokenizer.decode(seq[0], skip_special_tokens=True)
        pred_boxed = extract_boxed(text)

        bleu = compute_bleu(
            normalize_answer(true),
            normalize_answer(pred_boxed)
        )

        # сохраняем baseline
        if alpha == 0:
            bleu_0 = bleu

        # считаем delta
        delta_bleu = bleu - bleu_0 if bleu_0 is not None else 0.0

        results.append({
            "equation": eq,
            "true": true,
            "alpha": alpha,
            "steering_on": int(alpha != 0),
            "prediction": pred_boxed,
            "bleu": bleu,
            "bleu_baseline": bleu_0 if bleu_0 is not None else 0.0,
            "delta_bleu": delta_bleu
        })


# ------------------------------------------------------------
# SAVE RESULTS
# ------------------------------------------------------------
print("Saving results...")

df_out = pd.DataFrame(results)
df_out.to_excel(OUT_PATH, index=False)

print("Done.")